# Notebook 03 — Convert to Tableau long format

**Purpose:** Reshape policy lever outputs into long format tables that are easy to chart in Tableau.

**Inputs:**
- `tableau_policy_lever_impacts_strategic.csv`

**Outputs:**
- `outputs/tables/tableau_lever_sensitivity_long.csv`
- `outputs/tables/tableau_elasticity_heatmap.csv`


In [5]:
import pandas as pd

# Load your existing lever dataset
df = pd.read_csv("outputs/tables/tableau_policy_lever_impacts_strategic.csv")

# Define the components that we want to break into waterfall bars
components = {
    "ΔHighAI (Adoption Uplift)": "delta_highai",
    "ΔTools (Productivity Uplift)": "delta_toolcount",
    "ΔJob Satisfaction Uplift": "delta_jobsat",
    "ΔAI Threat (Penalty)": "delta_aithreat",
    "ΔSO Friction (Penalty)": "delta_sofriction"
}

# Build long-form table
rows = []

for _, r in df.iterrows():
    for comp_label, comp_col in components.items():
        rows.append({
            "Policy Lever": r["lever_pretty"],
            "Component": comp_label,
            "Value": r[comp_col],
            "Impact Score": r["impact_score"],
            "Strategic Risk": r["strategic_risk"]
        })

df_long = pd.DataFrame(rows)

# Save for Tableau
df_long.to_csv("outputs/tables/tableau_lever_sensitivity_long.csv", index=False)

df_long.head()


,Policy Lever,Component,Value,Impact Score,Strategic Risk
0,Uses AI Agents,ΔHighAI (Adoption Uplift),0.456112,0.531276,High Risk
1,Uses AI Agents,ΔTools (Productivity Uplift),0.053524,0.531276,High Risk
2,Uses AI Agents,ΔJob Satisfaction Uplift,0.113078,0.531276,High Risk
3,Uses AI Agents,ΔAI Threat (Penalty),0.027122,0.531276,High Risk
4,Uses AI Agents,ΔSO Friction (Penalty),0.152685,0.531276,High Risk


In [7]:
import pandas as pd
import numpy as np

# === LOAD BOTH DATASETS ===
df_levers = pd.read_csv("tableau_policy_lever_impacts.csv")
full = pd.read_csv("so2025_tableau_clean_final.csv")

# Ensure HighAI exists and is numeric
full["HighAI"] = full["HighAI"].fillna(0).astype(int)

# === DEFINE COHORT DIMENSIONS ===
cohort_columns = [
    "DevType_bucket",
    "ExpBand",
    "OrgSize_clean",
    "RemoteWork_clean"
]

rows = []

# === LOOP THROUGH EACH POLICY LEVER ===
for _, lev in df_levers.iterrows():
    lever_name = lev["lever_pretty"]
    dH = lev["delta_highai"]  # Marginal adoption uplift from the policy lever
    
    # Loop through cohort dimensions
    for col in cohort_columns:
        
        # Unique cohort values (avoid NaN)
        for val in full[col].dropna().unique():
            
            subset = full[full[col] == val]
            
            if subset.shape[0] == 0:
                continue
            
            base_rate = subset["HighAI"].mean()
            
            # Avoid division by zero
            elasticity = dH / max(base_rate, 0.0001)
            
            rows.append({
                "Policy Lever": lever_name,
                "Cohort Dimension": col,
                "Cohort Value": val,
                "Base HighAI Rate": base_rate,
                "Marginal ΔHighAI": dH,
                "Elasticity": elasticity
            })

# === FINAL DATAFRAME ===
df_elasticity = pd.DataFrame(rows)

# === SAVE FOR TABLEAU ===
df_elasticity.to_csv("outputs/tables/tableau_elasticity_heatmap.csv", index=False)

df_elasticity.head()


,Policy Lever,Cohort Dimension,Cohort Value,Base HighAI Rate,Marginal ΔHighAI,Elasticity
0,Uses AI Agents,DevType_bucket,Mobile,0.251980,0.456112,1.810112
1,Uses AI Agents,DevType_bucket,Web/Fullstack,0.231884,0.456112,1.966982
2,Uses AI Agents,DevType_bucket,Other/Unknown,0.218050,0.456112,2.091778
3,Uses AI Agents,DevType_bucket,Data/ML,0.278777,0.456112,1.636117
4,Uses AI Agents,DevType_bucket,Security,0.230352,0.456112,1.980061
